In [0]:
-- ============================================================================
-- BRIGHT MOTORS - CAR SALES DATA PROCESSING (FINAL VERSION)
-- Task 2: Data Processing in Snowflake
-- ============================================================================

-- 1. DATA QUALITY CHECK
----------------------------------------

SELECT * 
FROM brightlearn.case_study.bright_motor 
LIMIT 10;

SELECT DISTINCT make, body, transmission, state 
FROM brightlearn.case_study.bright_motor;

SELECT vin, COUNT(*) AS duplicate_count
FROM brightlearn.case_study.bright_motor
GROUP BY vin HAVING COUNT(*) > 1;

SELECT * 
FROM brightlearn.case_study.bright_motor
WHERE sellingprice IS NULL 
    OR make IS NULL 
    OR model IS NULL;

-- Check case inconsistencies
SELECT LOWER(body) AS body_normalized,
       COUNT(DISTINCT body) AS case_variants,
       STRING_AGG(DISTINCT body, ', ') AS all_variants,
       COUNT(*) AS total_records
FROM brightlearn.case_study.bright_motor
GROUP BY LOWER(body) HAVING COUNT(DISTINCT body) > 1;

SET legacy_time_parser_policy = LEGACY;

-- ============================================================================
-- 2. DATA CLEANING & FINAL TABLE
-- ============================================================================

CREATE OR REPLACE TABLE brightlearn.case_study.car_sales AS
SELECT

    -- CORE FIELDS (case-normalized + NULL handling)
    year,
    UPPER(COALESCE(make, 'Unknown')) AS make,
    UPPER(COALESCE(model, 'Unknown')) AS model,
    UPPER(COALESCE(trim, 'Unknown')) AS trim,
    UPPER(COALESCE(body, 'Unknown')) AS body,
    UPPER(COALESCE(transmission, 'Unknown')) AS transmission,
    vin,
    UPPER(COALESCE(state, 'Unknown')) AS state,

    condition,
    odometer,
    UPPER(COALESCE(color, 'Unknown')) AS color,
    UPPER(COALESCE(interior, 'Unknown')) AS interior,
    UPPER(COALESCE(seller, 'Unknown')) AS seller,

    -- NUMERIC CONVERSIONS (text-based prices to numeric)
    CAST(sellingprice AS DOUBLE) AS selling_price,
    CAST(mmr AS DOUBLE) AS mmr,

    -- DATE CONVERSION & TIME PERIOD GROUPING
    TO_DATE(TO_TIMESTAMP(saledate, 'EEE MMM dd yyyy HH:mm:ss')) AS sale_date,
    YEAR(TO_TIMESTAMP(saledate, 'EEE MMM dd yyyy HH:mm:ss')) AS sale_year,
    MONTH(TO_TIMESTAMP(saledate, 'EEE MMM dd yyyy HH:mm:ss')) AS sale_month,
    QUARTER(TO_TIMESTAMP(saledate, 'EEE MMM dd yyyy HH:mm:ss')) AS sale_quarter,

    -- CAR AGE
    YEAR(CURRENT_DATE()) - year AS car_age,

    -- UNITS SOLD & TOTAL REVENUE
    1 AS units_sold,
    CAST(sellingprice AS DOUBLE) AS total_revenue,

    -- PROFIT
    CAST(sellingprice AS DOUBLE) - CAST(mmr AS DOUBLE) AS profit,

    -- PROFIT MARGIN
    ((CAST(sellingprice AS DOUBLE) - CAST(mmr AS DOUBLE)) / CAST(sellingprice AS DOUBLE)) * 100 AS profit_margin,

    -- PROFIT TIER CATEGORIZATION
    CASE 
        WHEN ((CAST(sellingprice AS DOUBLE) - CAST(mmr AS DOUBLE)) / CAST(sellingprice AS DOUBLE)) * 100 >= 20 
            THEN 'High Margin'
        WHEN ((CAST(sellingprice AS DOUBLE) - CAST(mmr AS DOUBLE)) / CAST(sellingprice AS DOUBLE)) * 100 >= 10 
            THEN 'Medium Margin'
        ELSE 'Low Margin'
    END AS profit_tier,

    -- MILEAGE BAND (balanced: 30K / 75K split, nulls handled, no invalid category)
    CASE 
        WHEN odometer IS NULL THEN 'Unknown'
        WHEN odometer < 30000 THEN 'Low Mileage'
        WHEN odometer < 75000 THEN 'Medium Mileage'
        ELSE 'High Mileage'
    END AS mileage_band,

    -- PRICE VS MARKET (~20K At Market with ±$200 buffer)
    CASE 
        WHEN ABS(CAST(sellingprice AS DOUBLE) - CAST(mmr AS DOUBLE)) <= 200 THEN 'At Market'
        WHEN CAST(sellingprice AS DOUBLE) > CAST(mmr AS DOUBLE) THEN 'Above Market'
        ELSE 'Below Market'
    END AS price_vs_market

FROM brightlearn.case_study.bright_motor;


-- ============================================================================
-- 3. DATA OVERVIEW
-- ============================================================================

SELECT * 
FROM brightlearn.case_study.car_sales;

SELECT COUNT(*) AS total_sales FROM brightlearn.case_study.car_sales; --- 558811 total sales
SELECT COUNT(DISTINCT make) AS total_brands FROM brightlearn.case_study.car_sales; --- 67 brands
SELECT COUNT(DISTINCT model) AS total_models FROM brightlearn.case_study.car_sales; --- 852 models


-- ============================================================================
-- 4. REVENUE ANALYSIS
-- ============================================================================

SELECT make, SUM(total_revenue) AS total_revenue, COUNT(*) AS units_sold
FROM brightlearn.case_study.car_sales
GROUP BY make ORDER BY total_revenue DESC; --- FORD, CHEVROLET, NISSAN

SELECT make, model, COUNT(*) AS units_sold, SUM(total_revenue) AS total_revenue
FROM brightlearn.case_study.car_sales
GROUP BY make, model ORDER BY units_sold DESC; --- NISSAN   ALTIMA, FORD    F-150, FORD FUSION


-- ============================================================================
-- 5. PRODUCT ANALYSIS
-- ============================================================================

SELECT body, COUNT(*) AS total_sales, SUM(total_revenue) AS total_revenue
FROM brightlearn.case_study.car_sales
GROUP BY body ORDER BY total_sales DESC; --- SEDAN, SUV, HATCHBACK

SELECT transmission, COUNT(*) AS total_sales
FROM brightlearn.case_study.car_sales
GROUP BY transmission; --- AUTOMATIC, UNKNOWN, MANUAL

SELECT car_age, AVG(selling_price) AS avg_price, COUNT(*) AS units_sold
FROM brightlearn.case_study.car_sales
GROUP BY car_age ORDER BY car_age;


-- ============================================================================
-- 6. REGIONAL ANALYSIS
-- ============================================================================

SELECT state, COUNT(*) AS units_sold, SUM(total_revenue) AS total_revenue,
       AVG(profit_margin) AS avg_profit_margin
FROM brightlearn.case_study.car_sales
GROUP BY state ORDER BY total_revenue DESC;


-- ============================================================================
-- 7. SALES TREND ANALYSIS (Time Period Grouping)
-- ============================================================================

SELECT sale_year, COUNT(*) AS units_sold, SUM(total_revenue) AS total_revenue,
       AVG(profit_margin) AS avg_margin
FROM brightlearn.case_study.car_sales
GROUP BY sale_year ORDER BY sale_year;

SELECT sale_year, sale_quarter, COUNT(*) AS units_sold,
       SUM(total_revenue) AS total_revenue, AVG(profit_margin) AS avg_margin
FROM brightlearn.case_study.car_sales
GROUP BY sale_year, sale_quarter ORDER BY sale_year, sale_quarter;

SELECT sale_year, sale_month, COUNT(*) AS units_sold,
       SUM(total_revenue) AS total_revenue, AVG(profit_margin) AS avg_margin
FROM brightlearn.case_study.car_sales
GROUP BY sale_year, sale_month ORDER BY sale_year, sale_month;


-- ============================================================================
-- 8. PRICING & PROFITABILITY ANALYSIS
-- ============================================================================

SELECT make, AVG(profit_margin) AS avg_margin, COUNT(*) AS units_sold
FROM brightlearn.case_study.car_sales
GROUP BY make ORDER BY avg_margin DESC;

SELECT profit_tier, COUNT(*) AS total_sales, SUM(total_revenue) AS total_revenue,
       AVG(profit_margin) AS avg_margin
FROM brightlearn.case_study.car_sales
GROUP BY profit_tier ORDER BY avg_margin DESC;

SELECT price_vs_market, COUNT(*) AS total_sales, AVG(profit) AS avg_profit,
       AVG(profit_margin) AS avg_margin
FROM brightlearn.case_study.car_sales
GROUP BY price_vs_market;


-- ============================================================================
-- 9. SELLER PERFORMANCE
-- ============================================================================

SELECT seller, COUNT(*) AS units_sold, SUM(total_revenue) AS total_revenue,
       AVG(profit_margin) AS avg_margin
FROM brightlearn.case_study.car_sales
GROUP BY seller ORDER BY total_revenue DESC;


-- ============================================================================
-- 10. ADVANCED INSIGHTS
-- ============================================================================

SELECT mileage_band, AVG(selling_price) AS avg_price, COUNT(*) AS units_sold,
       AVG(profit_margin) AS avg_margin
FROM brightlearn.case_study.car_sales
GROUP BY mileage_band;

SELECT make, model, AVG(profit) AS avg_profit, AVG(profit_margin) AS avg_margin,
       COUNT(*) AS units_sold
FROM brightlearn.case_study.car_sales
GROUP BY make, model ORDER BY avg_profit DESC;

SELECT make, model, profit_tier, COUNT(*) AS units_sold,
       AVG(profit_margin) AS avg_margin
FROM brightlearn.case_study.car_sales
WHERE profit_tier = 'High Margin'
GROUP BY make, model, profit_tier
ORDER BY avg_margin DESC LIMIT 20;

-- ============================================================================
-- END OF SCRIPT
-- ============================================================================
